In [1]:
import pandas as pd
import numpy as np
import sklearn
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import root_mean_squared_error, mean_squared_error, mean_absolute_error, r2_score
from typing import Tuple, List, Any
from lightgbm import LGBMRegressor

import warnings, os
import mysql.connector as mysql
warnings.filterwarnings('ignore')
username = os.environ['MYSQL_user']
password = os.environ['MYSQL_password']
DB = mysql.connect(host = "localhost", user = username, passwd = password, database = "AIRBNB")
cursor = DB.cursor(buffered=True)
SEED = 17
compare_metric_name = 'RMSE'

## Utils

In [2]:
def read_table_from_db(table_name):
    df = pd.read_sql(f'SELECT * FROM {table_name}', con=DB)
    for col in df.columns:
        if len(df[col].unique()) == 2 or (df[col].dtype == 'object' and len(df[col].unique()) < 10):
            df[col] = df[col].astype('category')
    return df

In [3]:
def perform_cv(X: pd.DataFrame, y: pd.Series, algorithm: Any, cv: sklearn.model_selection = KFold(n_splits=5, shuffle=True, random_state=SEED), metric: sklearn.metrics = root_mean_squared_error) -> Tuple[List[float], List[float]]:
    """
    Perform cross-validation and return list of scores
    
    Args:
        X (pd.DataFrame): input data
        y (pd.Series): target data
        algorithm (Any): algorithm to use for training and prediction
        cv (sklearn.model_selection, default=KFold(n_splits=5, shuffle=True, random_state=SEED)): cross-validation strategy
        metric (sklearn.metrics, default=root_mean_squared_error): metric to use for evaluation
    
    Returns:
        Tuple[List[float], List[float]]: Tuple of lists of train and validation scores
    """
    train_scores, validation_scores = [], []
    for train_idx, val_idx in cv.split(X, y):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
        algorithm.fit(X_train, y_train)
        y_train_pred = algorithm.predict(X_train)
        y_val_pred = algorithm.predict(X_val)
        train_scores.append(metric(y_train, y_train_pred))
        validation_scores.append(metric(y_val, y_val_pred))
    return train_scores, validation_scores

def evaluation(X_train: pd.DataFrame, y_train: pd.Series, X_test: pd.DataFrame, y_test: pd.Series, algorithm: Any, metric: sklearn.metrics = root_mean_squared_error) -> Tuple[float, float, np.ndarray]:
    """
    Train the algorithm on the train data and evaluate on the train and test data
    
    Args:
        X_train (pd.DataFrame): input train data
        y_train (pd.Series): target train data
        X_test (pd.DataFrame): input test data
        y_test (pd.Series): target test data
        algorithm (Any): algorithm to use for training and prediction
        metric (sklearn.metrics, default=root_mean_squared_error): metric to use for evaluation
    
    Returns:
        Tuple[float, float, np.ndarray]: train_score, test_score, predictions on test data
    """
    algorithm.fit(X_train, y_train)
    y_train_pred = algorithm.predict(X_train)
    y_test_pred = algorithm.predict(X_test)
    train_results = metric(y_train, y_train_pred)
    test_results = metric(y_test, y_test_pred)
    return train_results, test_results, y_test_pred

# Load dataset

In [4]:
data = read_table_from_db('airbnb_data')
target_feature = 'log_price'
image_features = ["number_of_images", "number_of_bedroom_images", "number_of_bathroom_images", "number_of_living_room_images", "number_of_kitchen_images", "number_of_dining_room_images", "number_of_outside_building_images", "number_of_urban_environment_images", "number_of_other_images", "avg_warmth_hue", "avg_saturation", "avg_brightness", "avg_contrast_brightness", "avg_clarity"]
radius_meters = 100
location_features = ['distance_to_nearest_crime_m', f'number_of_crimes_within_{radius_meters}m', f'average_offence_weight_within_{radius_meters}m',
                    'distance_to_nearest_bus_stop_m', f'number_of_bus_stops_within_{radius_meters}m',
                    'distance_to_nearest_subway_station_m', f'number_of_subway_stations_within_{radius_meters}m',
                    'distance_to_nearest_restaurant_m', f'number_of_restaurants_within_{radius_meters}m',
                    'distance_to_nearest_education_institution_m', f'number_of_education_institutions_within_{radius_meters}m',
                    'distance_to_nearest_cultural_institution_m', f'number_of_cultural_institutions_within_{radius_meters}m',
                    'distance_to_nearest_recreation_point_m', f'number_of_recreation_points_within_{radius_meters}m',
                    'distance_to_nearest_religious_institution_m', f'number_of_religious_institutions_within_{radius_meters}m',
                    'distance_to_nearest_health_institution_m', f'number_of_health_institutions_within_{radius_meters}m',
                    'distance_to_nearest_main_attraction_m']
metadata_features = list(set(data.columns) - set(location_features) - set(image_features) - {target_feature})
data = data[metadata_features + [target_feature]]
data

,minimum_nights,bedrooms,review_scores_rating,district,maximum_nights,root_number_of_reviews,amenity_air_conditioning,time_since_first_review,time_since_last_review,amenity_smoke_alarm,...,availability_30,amenity_bed_linens,host_verification_email,review_scores_accuracy,amenity_iron,amenity_cooking_basics,instant_bookable,root_calculated_host_listings_count_shared_rooms,amenity_kitchen,log_price
0,30,0.0,91-95%,Manhattan,1125,7.00000,1,3 years and more,1 year and more,1,...,0,1,1,91-95%,1,1,0,0.0,1,5.48064
1,30,1.0,91-95%,Brooklyn,150,7.07107,1,3 years and more,1 year and more,0,...,0,0,1,81-90%,0,0,0,0.0,1,4.27667
2,30,2.0,91-95%,Brooklyn,730,13.92840,1,3 years and more,1-2 months,1,...,0,1,1,91-95%,1,1,0,0.0,1,4.39445
3,30,1.0,96-100%,Manhattan,180,1.00000,0,2-3 years,1 year and more,1,...,30,0,1,96-100%,0,0,0,0.0,1,4.17439
4,30,1.0,96-100%,Manhattan,365,15.74800,1,3 years and more,0-2 weeks,1,...,0,0,1,96-100%,0,0,0,0.0,1,4.17439
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19454,30,1.0,unknown,Brooklyn,180,0.00000,1,unknown,unknown,1,...,22,1,1,unknown,0,1,1,2.0,1,4.06044
19455,30,1.0,unknown,Brooklyn,365,0.00000,1,unknown,unknown,1,...,1,1,1,unknown,1,1,0,0.0,1,4.33073
19456,30,1.0,unknown,Staten Island,365,0.00000,0,unknown,unknown,1,...,0,1,1,unknown,1,1,0,0.0,1,3.71357
19457,30,1.0,unknown,Brooklyn,365,0.00000,1,unknown,unknown,1,...,30,0,0,unknown,0,0,0,0.0,0,4.75359


## Split dataset

In [5]:
train_data, test_data = train_test_split(data, test_size=0.2, random_state=SEED)

## Base model

In [6]:
model = LGBMRegressor(random_state=SEED, verbose=-1, linear_tree=True)
train_scores, validation_scores = perform_cv(train_data[metadata_features], train_data[target_feature], model, cv=KFold(n_splits=5, shuffle=True, random_state=SEED), metric=root_mean_squared_error)
print(f"Train {compare_metric_name}: {np.mean(train_scores):.4f} +- {np.std(train_scores):.4f}")
print(f"Validation {compare_metric_name}: {np.mean(validation_scores):.4f} +- {np.std(validation_scores):.4f}")

Train RMSE: 0.2739 +- 0.0015
Validation RMSE: 0.3927 +- 0.0759


In [ ]:
X_train = train_data[metadata_features]
y_train = train_data[target_feature]
X_test = test_data[metadata_features]
y_test = test_data[target_feature]
model = LGBMRegressor(random_state=SEED, verbose=-1, n_jobs=-1, objective='regression', metric=compare_metric_name, linear_tree=True)
model.fit(X_train, y_train)
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)
metrics = {
    'R2': r2_score,
    'R2_adj': lambda y_true, y_pred: 1 - (1 - r2_score(y_true, y_pred)) * (len(y_true) - 1) / (len(y_true) - X_train.shape[1] - 1),
    'MSE': mean_squared_error,
    'RMSE': root_mean_squared_error,
    'MAE': mean_absolute_error,
}
results = {}
for metric_name, metric in metrics.items():
    train_score = metric(y_train, y_train_pred)
    test_score = metric(y_test, y_test_pred)
    results[metric_name] = {
        'train': train_score,
        'test': test_score
    }
df = pd.DataFrame(results).T
df

,train,test
R2,0.853687,-2.483896
R2_adj,0.853687,-2.483896
MSE,0.079469,1.965822
RMSE,0.281902,1.402078
MAE,0.212792,0.281373
